# Метеоданные заповедника · интерактивный EDA

Открой ноутбук в Google Colab. Запусти первые три ячейки по порядку и выбери **полный ZIP организаторов** `meteo_hack_dataset.zip` или `meteo_hack_dataset (1).zip`. Распаковывать ZIP не нужно. Дальше выбери нужный раздел и нажми ▶. После изменения фильтров запусти соответствующую ячейку снова.

**Для защиты** начни с Кр1 и давления за апрель 2024, затем покажи сравнение поля и поляны. На графиках можно приближать даты, отключать серии и наводить курсор на записи. Значения и время хранятся как в исходных данных.

In [ ]:
%pip -q install "pandas>=2.2,<4" "plotly>=6,<8"

In [ ]:
# @title Правила расчёта и графики
"""Rules and exploratory summaries for the reserve weather archive."""

from io import BytesIO
from itertools import groupby
from pathlib import PurePosixPath
from zipfile import ZipFile

import numpy as np
import pandas as pd


METRICS = {
    "t_air": ("Температура воздуха", "°C"),
    "rh": ("Относительная влажность", "%"),
    "wind_speed": ("Скорость ветра", "м/с"),
    "wind_gust": ("Порывы ветра", "м/с"),
    "wind_dir": ("Направление ветра", "°"),
    "soil_05": ("Температура почвы на 5 см", "°C"),
    "soil_10": ("Температура почвы на 10 см", "°C"),
    "soil_20": ("Температура почвы на 20 см", "°C"),
    "pressure_hpa": ("Атмосферное давление", "гПа"),
}
STATIONS = {"ame60": "АМЕ-60", "kr1": "Кр1 · Поляна", "kr2": "Кр2 · Поле"}
MONTHS = ("Янв", "Фев", "Мар", "Апр", "Май", "Июн", "Июл", "Авг", "Сен", "Окт", "Ноя", "Дек")


def parse_observations(stream, filename):
    frame = pd.read_csv(stream, low_memory=False)
    metrics = [metric for metric in METRICS if metric in frame]
    if "datetime" not in frame or not metrics or frame.empty:
        raise ValueError(f"В файле {filename} нужны datetime, хотя бы один известный показатель и записи.")
    frame["datetime"] = pd.to_datetime(frame["datetime"], format="%Y-%m-%d %H:%M")
    if frame["datetime"].isna().any():
        raise ValueError(f"В файле {filename} найдены пустые даты.")
    frame["qc_flag"] = frame["qc_flag"].fillna("").astype(str) if "qc_flag" in frame else ""
    frame[metrics] = frame[metrics].apply(pd.to_numeric)
    frame["source"] = filename
    frame["csv_line"] = np.arange(2, len(frame) + 2)
    return frame[["datetime", *metrics, "qc_flag", "source", "csv_line"]]


class MeteoArchive:
    def __init__(self, observations, reference):
        self.observations = observations
        self.reference = reference
        self.labels = dict(STATIONS)
        self.profiles = {"ame60": 72, "kr1": 18, "kr2": 18}

    @classmethod
    def read(cls, source):
        frames = {station: [] for station in STATIONS}
        with ZipFile(source) as archive:
            ref_path = next((p for p in archive.namelist() if PurePosixPath(p).name == "coverage_by_month.csv"), None)
            if ref_path is None:
                raise ValueError("Выбери полный ZIP организаторов с coverage_by_month.csv.")
            with archive.open(ref_path) as stream:
                reference = pd.read_csv(stream)
            data_dir = PurePosixPath(ref_path).parent / "data"
            for name in sorted(archive.namelist()):
                path = PurePosixPath(name)
                station = path.stem.rsplit("_", 1)[0]
                if path.suffix == ".csv" and path.is_relative_to(data_dir) and station in frames:
                    with archive.open(name) as stream:
                        frames[station].append(parse_observations(stream, path.name))
        if not all(frames.values()):
            raise ValueError("В архиве нужны файлы АМЕ-60, Кр1 и Кр2.")
        observations = {
            station: pd.concat(parts, ignore_index=True).sort_values("datetime", kind="stable")
            for station, parts in frames.items()
        }
        return cls(observations, reference)

    def add_station(self, filename, content, name, minimum):
        if minimum not in (72, 18) or not name.strip():
            raise ValueError("Укажи название станции и профиль полноты.")
        frame = parse_observations(BytesIO(content), filename)
        frame = frame.drop_duplicates(subset=frame.columns.difference(["source", "csv_line"]))
        conflicts = frame.duplicated("datetime", keep=False) & ~flag_token(frame["qc_flag"], "dup_conflict")
        frame.loc[conflicts, "qc_flag"] = frame.loc[conflicts, "qc_flag"].map(
            lambda flags: f"{flags};dup_conflict" if flags else "dup_conflict"
        )
        self.observations["extra"] = frame.sort_values("datetime", kind="stable")
        self.labels["extra"] = name.strip()
        self.profiles["extra"] = minimum
        return len(frame)


def flag_token(flags, name, *, parameter=False):
    ending = ":" if parameter else "(?:;|$)"
    return flags.str.contains(f"(?:^|;){name}{ending}", regex=True, na=False)


def period_bounds(start, end):
    start, end = pd.Timestamp(start).normalize(), pd.Timestamp(end).normalize()
    if end < start:
        raise ValueError("Дата окончания должна быть не раньше даты начала.")
    return start, end


def observation_rows(archive, station, metric, start, end):
    frame = archive.observations[station]
    start, end = period_bounds(start, end)
    in_period = frame["datetime"].ge(start) & frame["datetime"].lt(end + pd.Timedelta(days=1))
    rows = frame.loc[in_period, ["datetime", "qc_flag", "source", "csv_line"]].copy()
    rows["value"] = frame.loc[in_period, metric] if metric in frame else np.nan
    rows["day"] = rows["datetime"].dt.normalize()
    rows["hour"] = rows["datetime"].dt.hour
    rows["conflict"] = flag_token(rows["qc_flag"], "dup_conflict")
    bad_direction = station == "ame60" and metric == "wind_dir"
    blocked = rows["datetime"].dt.year.eq(2021) if bad_direction else False
    rows["metric_flag"] = flag_token(rows["qc_flag"], metric, parameter=True)
    missing = ~np.isfinite(rows["value"])
    rows["status"] = np.select(
        [rows["conflict"], np.broadcast_to(blocked, len(rows)), rows["metric_flag"], missing],
        ["Конфликт записей", "Направление ветра за 2021 год исключено", "Флаг показателя", "Нет значения"],
        default="Пригодный замер",
    )
    rows["valid"] = rows["status"].eq("Пригодный замер")
    return rows


def daily_coverage(rows, minimum, start, end, *, scalar=True):
    calendar = pd.date_range(start, end, freq="D", name="day")
    clean = rows.loc[rows["valid"]].groupby("day")
    station = rows.loc[~rows["conflict"]].groupby("day")
    slots = "datetime" if minimum == 72 else "hour"
    daily = pd.DataFrame(index=calendar)
    daily["rows"] = rows.groupby("day").size().reindex(calendar, fill_value=0)
    daily["valid_readings"] = clean.size().reindex(calendar, fill_value=0)
    daily["station_slots"] = station[slots].nunique().reindex(calendar, fill_value=0)
    daily["metric_slots"] = clean[slots].nunique().reindex(calendar, fill_value=0)
    daily["station_ok"] = daily["station_slots"].ge(minimum)
    daily["metric_ok"] = daily["metric_slots"].ge(minimum)
    if scalar:
        for stat in ("mean", "min", "max"):
            daily[stat] = clean["value"].agg(stat).reindex(calendar).where(daily["metric_ok"])
    return daily


def longest_gap(eligible):
    return max((sum(1 for _ in run) for present, run in groupby(eligible) if not present), default=0)


def period_summary(daily, start, end, *, by_month, scalar=True):
    if by_month:
        grouped = daily.groupby(daily.index.to_period("M"))
    else:
        offset = np.minimum((daily.index.day - 1) // 10, 2) * 10
        starts = daily.index.to_period("M").to_timestamp() + pd.to_timedelta(offset, unit="D")
        grouped = daily.groupby(starts)
    result = []
    for _, group in grouped:
        first, last = group.index[0], group.index[-1]
        if last < start or first > end:
            continue
        assumed_days = len(group)
        metric_days = int(group["metric_ok"].sum())
        station_days = int(group["station_ok"].sum())
        gap = longest_gap(group["metric_ok"])
        station_gap = longest_gap(group["station_ok"])
        full_selection = start <= first and end >= last
        month_ok = assumed_days - metric_days <= 10 and gap < 5
        station_month_ok = assumed_days - station_days <= 10 and station_gap < 5
        eligible = bool(metric_days and full_selection and (month_ok or not by_month))
        result.append({
            "start": first, "end": last, "days": assumed_days, "station_days": station_days,
            "metric_days": metric_days, "gap": gap, "eligible": eligible,
            "station_month_ok": station_month_ok, "full_selection": full_selection,
            "mean": group["mean"].mean() if eligible and scalar else np.nan,
        })
    return pd.DataFrame(result)


def analyze(archive, station, metric, start, end):
    start, end = period_bounds(start, end)
    first_month = start.to_period("M").start_time
    last_month = end.to_period("M").end_time.normalize()
    month_rows = observation_rows(archive, station, metric, first_month, last_month)
    scalar = metric != "wind_dir"
    full_days = daily_coverage(month_rows, archive.profiles[station], first_month, last_month, scalar=scalar)
    return (
        month_rows.loc[month_rows["day"].between(start, end)],
        full_days.loc[start:end],
        period_summary(full_days, start, end, by_month=True, scalar=scalar),
        period_summary(full_days, start, end, by_month=False, scalar=scalar),
    )


def reference_coverage(archive):
    actual = []
    for station in STATIONS:
        reference = archive.reference.loc[archive.reference["station"].str.lower().eq(station)]
        first, last = int(reference["year"].min()), int(reference["year"].max())
        _, _, monthly, _ = analyze(archive, station, "t_air", f"{first}-01-01", f"{last}-12-31")
        actual.append(monthly.assign(station=station, year=monthly.start.dt.year, month=monthly.start.dt.month))
    combined = pd.concat(actual, ignore_index=True)
    expected = archive.reference.assign(station=archive.reference["station"].str.lower())
    checked = expected.merge(combined, on=["station", "year", "month"], validate="one_to_one", how="left")
    matches = checked["valid_days"].eq(checked["station_days"]) & checked["month_valid"].eq(checked["station_month_ok"])
    return checked, checked.loc[~matches]


def compare_temperature(archive, year, month):
    start, end = f"{year}-01-01", f"{year}-12-31"
    kr1, kr2 = (analyze(archive, name, "t_air", start, end) for name in ("kr1", "kr2"))
    combined = pd.concat([kr1[1]["mean"].rename("clearing"), kr2[1]["mean"].rename("field")], axis=1)
    available_months = kr1[2].loc[kr1[2]["eligible"], "start"].dt.month
    available_months = set(available_months) & set(kr2[2].loc[kr2[2]["eligible"], "start"].dt.month)
    combined["delta"] = combined["field"] - combined["clearing"]
    monthly = combined["delta"].resample("MS").agg(["mean", "count"])
    monthly["eligible"] = monthly.index.month.isin(available_months)
    monthly["mean"] = monthly["mean"].where(monthly["eligible"])
    paired = combined.loc[combined.index.month == month].dropna()

    def per_hour(rows):
        return rows.loc[rows["valid"] & rows["day"].isin(paired.index)].groupby(["day", "hour"])["value"].mean()

    by_hour = pd.concat([per_hour(kr1[0]).rename("clearing"), per_hour(kr2[0]).rename("field")], axis=1).dropna()
    by_hour["delta"] = by_hour["field"] - by_hour["clearing"]
    hourly = by_hour.groupby("hour").agg(
        clearing=("clearing", "mean"), field=("field", "mean"), delta=("delta", "mean"), pairs=("delta", "size"),
    ).reindex(range(24))
    return monthly, paired, hourly


def historical_context(archive, station, metric, month, first, last):
    if metric == "wind_dir":
        raise ValueError("Для направления ветра смотри распределение по секторам.")
    if first > last:
        raise ValueError("Последний год должен быть не раньше первого.")
    _, _, monthly, _ = analyze(archive, station, metric, f"{first}-01-01", f"{last}-12-31")
    result = monthly.loc[monthly["start"].dt.month.eq(month)].copy()
    result["year"] = result["start"].dt.year
    mean = result["mean"].mean()
    result["difference"] = result["mean"] - mean
    available = result.dropna(subset=["mean"])
    result["trend"] = np.nan
    slope = None
    if len(available) >= 3:
        x = available["year"] - first
        slope, intercept = np.polyfit(x, available["mean"], 1)
        result.loc[available.index, "trend"] = slope * x + intercept
    return result, mean, len(available), None if slope is None else 10 * slope

"""Interactive charts shared by the Colab notebook and offline defense report."""

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots


GREEN = "#117666"
RED = "#a83629"
BLUE = "#355b93"
NEUTRAL = "#52645e"


def finish(figure, title, *, height=430):
    figure.update_layout(
        template="simple_white", title=dict(text=title, x=0.025), height=height, autosize=True,
        font=dict(family="Arial, sans-serif", size=13, color="#15322a"),
        margin=dict(l=60, r=32, t=75, b=70),
        legend=dict(orientation="h", y=-0.18, x=0), separators=", ", hoverlabel=dict(namelength=-1),
    )
    figure.update_yaxes(gridcolor="#e6eeea", zerolinecolor="#9fb2a9")
    return figure


def measurements_figure(rows, unit, title):
    plot = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.15,
                         subplot_titles=("Исходные замеры", "После проверки качества"))
    for valid, color, label in [(True, GREEN, "Пригодный замер"), (False, RED, "Исключённый замер")]:
        measured = rows.loc[rows["valid"].eq(valid) & rows["value"].notna()]
        if measured.empty:
            continue
        step = max(1, (len(measured) + 7499) // 7500)
        displayed = measured.iloc[::step]
        trace = go.Scatter(
            x=displayed["datetime"], y=displayed["value"], mode="markers", name=label,
            marker=dict(color=color, size=5, opacity=0.77, symbol="circle" if valid else "x"),
            customdata=displayed[["status", "qc_flag", "source", "csv_line"]].to_numpy(),
            hovertemplate="%{x|%d.%m.%Y %H:%M}<br>%{y:.2f} " + unit
            + "<br>%{customdata[0]}<br>%{customdata[1]}<br>%{customdata[2]} · строка %{customdata[3]}<extra></extra>",
        )
        plot.add_trace(trace, row=1, col=1)
        if valid:
            plot.add_trace(go.Scatter(trace, showlegend=False), row=2, col=1)
    if not plot.data:
        plot.add_annotation(text="За выбранный период нет числовых значений", showarrow=False, xref="paper", yref="paper", x=0.5, y=0.5)
    plot.update_yaxes(title_text=unit)
    plot.update_xaxes(title_text="Дата и время архива", row=2, col=1)
    return finish(plot, title, height=625)


def coverage_figure(daily, minimum, title):
    plot = go.Figure()
    for column, name, color in [("station_slots", "По станции", BLUE), ("metric_slots", "По показателю", GREEN)]:
        plot.add_bar(x=daily.index, y=daily[column], name=name, marker_color=color,
                     hovertemplate="%{x|%d.%m.%Y}<br>%{y} доступных интервалов<extra>" + name + "</extra>")
    plot.add_hline(y=minimum, line_dash="dash", line_color=RED, annotation_text=f"Порог {minimum}")
    plot.update_layout(barmode="group")
    plot.update_yaxes(title_text="Сроков за сутки" if minimum == 72 else "Часов за сутки", rangemode="tozero")
    plot.update_xaxes(title_text="Дата архива")
    return finish(plot, title)


def aggregates_figure(daily, monthly, ten_day, granularity, stat, unit, title):
    if granularity == "Сутки":
        points = daily[stat]
        name = {"mean": "Среднее", "min": "Минимум", "max": "Максимум"}[stat]
    else:
        points = (monthly if granularity == "Месяц" else ten_day).set_index("start")["mean"]
        name = "Среднее суточных средних"
    plot = go.Figure(go.Scatter(x=points.index, y=points, name=name, mode="lines+markers", connectgaps=False,
                                line=dict(color=GREEN), marker=dict(size=5),
                                hovertemplate="%{x|%d.%m.%Y}<br>%{y:.2f} " + unit + "<extra>" + name + "</extra>"))
    if not points.notna().any():
        plot.add_annotation(text="В этом периоде нет пригодных агрегатов", showarrow=False, xref="paper", yref="paper", x=0.5, y=0.5)
    plot.update_yaxes(title_text=unit)
    plot.update_xaxes(title_text="Начало периода")
    return finish(plot, title)


def station_coverage_figure(checked, labels, months):
    plot = make_subplots(rows=3, cols=1, vertical_spacing=0.12, subplot_titles=list(labels.values())[:3])
    for i, station in enumerate(list(labels)[:3], 1):
        source = checked[checked["station"].eq(station)].copy()
        source["percent"] = 100 * source["station_days"] / source["days"]
        years = range(int(source.year.min()), int(source.year.max()) + 1)
        grid = source.pivot(index="month", columns="year", values="percent").reindex(index=range(1, 13), columns=years)
        plot.add_trace(go.Heatmap(
            x=list(years), y=list(months), z=grid.to_numpy(), coloraxis="coloraxis",
            hovertemplate="%{y} %{x}<br>Пригодные дни %{z:.0f}%<extra></extra>",
        ), row=i, col=1)
        plot.update_yaxes(autorange="reversed", row=i, col=1)
        plot.update_xaxes(type="category", nticks=12, row=i, col=1)
    plot.update_layout(coloraxis=dict(colorscale=[[0, "#f4e8e0"], [1, GREEN]], cmin=0, cmax=100,
                                      colorbar=dict(title="Дни, %", len=0.72)))
    return finish(plot, "Покрытие наблюдений по месяцам", height=790)


def temperature_comparison_figure(monthly, hourly, months, month):
    plot = make_subplots(rows=2, cols=1, vertical_spacing=0.19,
                         subplot_titles=("Разница по месяцам", f"Средний суточный ход · {months[month - 1]}"))
    for y, x, hover_counts, row, name in [
        (monthly["mean"], [months[i - 1] for i in monthly.index.month], monthly["count"], 1, "Общих дней"),
        (hourly["delta"], [f"{h:02d}:00" for h in hourly.index], hourly["pairs"], 2, "Пар дней"),
    ]:
        plot.add_trace(go.Bar(x=x, y=y, customdata=hover_counts,
                              marker_color=[BLUE if val < 0 else GREEN for val in y], showlegend=False,
                              hovertemplate="%{x}<br>Кр2 − Кр1 %{y:+.2f} °C<br>" + name + " %{customdata:.0f}<extra></extra>"),
                       row=row, col=1)
    plot.update_yaxes(title_text="Кр2 − Кр1, °C")
    plot.update_xaxes(nticks=12, row=2, col=1)
    return finish(plot, "Температура на поле и поляне · 2025", height=625)


def history_figure(result, baseline, unit, title):
    plot = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.16,
                         subplot_titles=("Месячные средние", "Отклонение от среднего выбранных лет"))
    plot.add_trace(go.Scatter(x=result["year"], y=result["mean"], name="Пригодные месяцы", mode="lines+markers",
                              line=dict(color=GREEN), connectgaps=False), row=1, col=1)
    if result["trend"].notna().any():
        plot.add_trace(go.Scatter(x=result["year"], y=result["trend"], name="Описательный тренд", mode="lines",
                                  line=dict(color=BLUE, dash="dot")), row=1, col=1)
    if pd.notna(baseline):
        plot.add_hline(y=baseline, line_dash="dash", line_color=NEUTRAL, row=1, col=1)
    plot.add_trace(go.Bar(x=result["year"], y=result["difference"], name="Отклонение", showlegend=False,
                          marker_color=[BLUE if val < 0 else GREEN for val in result["difference"]]), row=2, col=1)
    plot.update_yaxes(title_text=unit)
    plot.update_xaxes(title_text="Год", tickformat="d", row=2, col=1)
    return finish(plot, title, height=595)


def wind_figure(rows, title):
    direction = rows.loc[rows["valid"] & rows["value"].between(0, 360), "value"]
    plot = go.Figure()
    if not direction.empty:
        sector = ((direction + 11.25) / 22.5).astype(int) % 16
        counts = sector.value_counts().reindex(range(16), fill_value=0)
        plot.add_barpolar(r=100 * counts / len(direction), theta=[i * 22.5 for i in range(16)],
                          width=19, marker_color=GREEN, customdata=counts,
                          hovertemplate="%{theta}°<br>%{r:.1f}% замеров<br>%{customdata} записей<extra></extra>")
    else:
        plot.add_annotation(text="Пригодных направлений за этот период нет", showarrow=False, x=0.5, y=0.5,
                            xref="paper", yref="paper")
    plot.update_layout(polar=dict(angularaxis=dict(direction="clockwise", rotation=90,
                                                   tickvals=list(range(0, 360, 45)),
                                                   ticktext=["С", "СВ", "В", "ЮВ", "Ю", "ЮЗ", "З", "СЗ"]),
                                  radialaxis=dict(ticksuffix="%")))
    return finish(plot, title)

"""Notebook views, source tables and the data exported for defense."""

import numpy as np
import pandas as pd



def show_inventory(archive):
    for station, table in archive.observations.items():
        earliest = table.datetime.min().date()
        latest = table.datetime.max().date()
        print(f"{archive.labels[station]} · {earliest:%d.%m.%Y} – {latest:%d.%m.%Y} · {len(table):,} записей")


def show_selection(archive, station, metric, start, end, view, detail_day=""):
    from IPython.display import display

    rows, daily, monthly, ten_day = analyze(archive, station, metric, start, end)
    label, unit = METRICS[metric]
    title = f"{archive.labels[station]} · {label}"
    print(f"Пригодные сутки · станция {daily.station_ok.sum()} из {len(daily)} · показатель {daily.metric_ok.sum()} из {len(daily)}")
    print(f"Записи · {len(rows)} всего · {rows.valid.sum()} пригодных по показателю")
    if len(rows) and not rows.valid.all():
        for reason, count in rows.loc[~rows.valid, "status"].value_counts().items():
            print(f"{reason} · {count}")
    if metric == "wind_dir" and view not in ("Замеры и флаги", "Полнота по дням"):
        fig = wind_figure(rows, f"Частоты направлений · {archive.labels[station]}")
        print("Секторы показывают долю пригодных замеров направления без деления по скорости ветра.")
    elif view == "Замеры и флаги":
        fig = measurements_figure(rows, unit, title)
        if len(rows) > 7500:
            print("На длинных интервалах точки могут отображаться с прореживанием. CSV содержит все строки.")
    elif view == "Полнота по дням":
        fig = coverage_figure(daily, archive.profiles[station], title)
    elif view == "Направления по секторам":
        raise ValueError("Выбери направление ветра, чтобы увидеть секторы.")
    else:
        granularity, stat = {
            "Среднее за сутки": ("Сутки", "mean"),
            "Минимум за сутки": ("Сутки", "min"),
            "Максимум за сутки": ("Сутки", "max"),
            "Среднее за декаду": ("Декада", "mean"),
            "Среднее за месяц": ("Месяц", "mean"),
        }[view]
        fig = aggregates_figure(daily, monthly, ten_day, granularity, stat, unit, f"{title} · {view.lower()}")
    fig.show(config={"displaylogo": False, "responsive": True})

    monthly = monthly.assign(
        month=monthly["start"].dt.strftime("%Y-%m"),
        reason=np.select(
            [~monthly["full_selection"], monthly["eligible"], monthly["metric_days"].eq(0)],
            ["Месяц выбран частично", "Проходит", "Нет пригодных суток"],
            default="Месяц не проходит порог полноты",
        ),
    )
    display(monthly[["month", "days", "station_days", "metric_days", "gap", "reason"]].rename(columns={
        "month": "Месяц", "days": "Дней", "station_days": "По станции", "metric_days": "По показателю",
        "gap": "Неполных дней подряд", "reason": "Статус для показателя",
    }).head(36))
    if len(monthly) > 36:
        print(f"Выведено 36 из {len(monthly)} месяцев. Таблица целиком доступна для скачивания.")

    detail = rows.loc[rows["day"].eq(pd.Timestamp(detail_day).normalize())] if detail_day else rows.loc[~rows["valid"]]
    if not detail.empty:
        print("Исходные записи выбранной даты" if detail_day else "Первые исключённые строки")
        display(detail[["datetime", "value", "status", "qc_flag", "source", "csv_line"]].head(25))
    elif detail_day:
        print("На выбранную дату в этом окне записи отсутствуют.")
    return rows, daily, monthly, ten_day


def archive_comparison(archive, month):
    from IPython.display import display

    monthly, paired, hourly = compare_temperature(archive, 2025, month)
    if not paired.empty:
        print(f"{MONTHS[month - 1]} · {len(paired)} общих пригодных дней · средняя разница Кр2 − Кр1 {paired.delta.mean():+.2f} °C")
    temperature_comparison_figure(monthly, hourly, MONTHS, month).show(config={"displaylogo": False, "responsive": True})
    display(hourly.rename(columns={"clearing": "Кр1, °C", "field": "Кр2, °C", "delta": "Разница, °C", "pairs": "Общих дней"}).round(3))


def show_history(archive, station, metric, month, first, last):
    result, baseline, years, slope = historical_context(archive, station, metric, month, first, last)
    label, unit = METRICS[metric]
    if years:
        print(f"Среднее за выбранный период {baseline:.2f} {unit} · пригодных лет {years} из {len(result)}")
    else:
        print("В выбранном месяце и диапазоне нет пригодных лет.")
    if slope is not None:
        print(f"Линейное изменение {slope:+.2f} {unit} за десятилетие · описательная оценка")
    history_figure(result, baseline, unit, f"{archive.labels[station]} · {label} · {MONTHS[month - 1]}").show(
        config={"displaylogo": False, "responsive": True}
    )
    return result

In [ ]:
# @title Открыть архив наблюдений
from os import environ
from zipfile import BadZipFile
import plotly.io as pio

archive_path = ""  # @param {type:"string"}
local_path = archive_path.strip() or environ.get("METEO_ARCHIVE")
if local_path:
    archive = MeteoArchive.read(local_path)
else:
    from google.colab import files

    uploaded = files.upload()
    selected = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(selected) != 1:
        raise ValueError("Выбери один ZIP из материалов хакатона.")
    try:
        archive = MeteoArchive.read(BytesIO(uploaded[selected[0]]))
    except BadZipFile as error:
        raise ValueError("Не удалось прочитать ZIP. Скачай исходный архив заново.") from error
    del uploaded
    pio.renderers.default = "colab"

show_inventory(archive)
print(f"Всего {sum(len(frame) for frame in archive.observations.values()):,} наблюдений")

## Станция, показатель и период

Первый запуск показывает давление на Кр1. В строке под графиком увидишь **28 из 30** пригодных суток по станции и **2 из 30** по давлению. Зелёные точки — пригодные замеры, красные крестики — исключённые. График после фильтрации сохраняет отдельные замеры с неполных суток. В сводки входят только сутки, прошедшие порог. Чтобы увидеть конкретную запись, наведи курсор на точку или укажи дату в `detail_day`.

Полноту станции и полноту отдельного показателя считаем раздельно. Порог по показателю применяется ко всем исследовательским сводкам в этом ноутбуке. Для давления Кр1 это часть задания, для остальных параметров это явно выбранный строгий режим EDA. Сутки считаются по 72 уникальным срокам АМЕ-60 или 18 уникальным часам Сокол М1. Суточное среднее использует все пригодные записи суток. Для декады или месяца усредняем пригодные суточные средние с одинаковым весом дней. Проверка месяца требует не более 10 неполных дней и максимум 4 подряд. Месяцы и декады, выбранные частично, остаются помечены как частичные.

In [ ]:
# @title Выбрать данные и график
station_label = "Кр1 · Поляна"  # @param ["АМЕ-60", "Кр1 · Поляна", "Кр2 · Поле", "Добавленная станция"]
metric_label = "Атмосферное давление"  # @param ["Температура воздуха", "Относительная влажность", "Скорость ветра", "Порывы ветра", "Направление ветра", "Температура почвы на 5 см", "Температура почвы на 10 см", "Температура почвы на 20 см", "Атмосферное давление"]
start = "2024-04-01"  # @param {type:"date"}
end = "2024-04-30"  # @param {type:"date"}
view = "Замеры и флаги"  # @param ["Замеры и флаги", "Полнота по дням", "Среднее за сутки", "Минимум за сутки", "Максимум за сутки", "Среднее за декаду", "Среднее за месяц", "Направления по секторам"]
detail_day = "2024-04-02"  # @param {type:"string"}

station = "extra" if station_label == "Добавленная станция" else next(code for code, label in STATIONS.items() if label == station_label)
metric = next(code for code, (label, unit) in METRICS.items() if label == metric_label)
if station not in archive.observations:
    raise ValueError("Сначала загрузи CSV в разделе «Добавить станцию для исследования».")
selected_rows, selected_daily, selected_monthly, selected_ten_day = show_selection(
    archive, station, metric, start, end, view, detail_day.strip()
)

## Где в архиве есть пригодные данные

Тепловая карта показывает долю пригодных суток по станции. Пустая ячейка означает, что месяц отсутствует в эталонной таблице. Проценты позволяют быстро найти сезон с хорошим покрытием и длинные пробелы в архиве.

In [ ]:
# @title Сверить покрытие с эталоном
checked, mismatches = reference_coverage(archive)
print(f"Совпали {len(checked) - len(mismatches)} из {len(checked)} строк coverage_by_month.csv")
if not mismatches.empty:
    display(mismatches)
station_coverage_figure(checked, archive.labels, MONTHS).show(config={"displaylogo": False})

## Поляна и поле · 2025

Разница везде считается как **Кр2 − Кр1**. Для месячного графика берём общие пригодные дни. Для суточного хода сравниваем средние внутри одного и того же часа одной и той же даты и показываем число пар. В июле средняя разница мала, а в отдельные часы станции расходятся заметнее. Часы подписаны по времени архива.

In [ ]:
# @title Выбрать месяц для суточного хода
month = 7  # @param {type:"slider", min:1, max:12, step:1}
archive_comparison(archive, month)

## Один месяц в разные годы

Сравниваем только одноимённые календарные месяцы одной станции. На графике видны пропуски и число лет в расчёте. При трёх и более пригодных годах показываем описательный линейный тренд за десятилетие. Эту оценку можно использовать для формулировки исследовательского вопроса.

In [ ]:
# @title Историческое сравнение и отклонения
history_station = "АМЕ-60"  # @param ["АМЕ-60", "Кр1 · Поляна", "Кр2 · Поле"]
history_metric = "Температура воздуха"  # @param ["Температура воздуха", "Относительная влажность", "Скорость ветра", "Порывы ветра", "Температура почвы на 5 см", "Температура почвы на 10 см", "Температура почвы на 20 см", "Атмосферное давление"]
month = 7  # @param {type:"slider", min:1, max:12, step:1}
first_year = 1992  # @param {type:"integer"}
last_year = 2021  # @param {type:"integer"}
station_code = next(code for code, label in STATIONS.items() if label == history_station)
metric_code = next(code for code, (label, unit) in METRICS.items() if label == history_metric)
history = show_history(archive, station_code, metric_code, month, first_year, last_year)

## Дополнительные действия

Добавленную станцию можно сразу открыть в фильтре основного графика. Для пробной загрузки нужен CSV в формате хакатона. Название, данные и профиль действуют в текущем сеансе Colab. Укажи профиль по частоте наблюдений, принятой для этой станции.

In [ ]:
# @title Добавить станцию для исследования
from google.colab import files

new_name = "Новая станция"  # @param {type:"string"}
profile = "18 часов"  # @param ["18 часов", "72 срока"]
batch = files.upload()
csvs = [name for name in batch if name.lower().endswith(".csv")]
if len(csvs) != 1:
    raise ValueError("Выбери один CSV в формате архива.")
name = csvs[0]
count = archive.add_station(name, batch[name], new_name, 18 if profile == "18 часов" else 72)
print(f"Добавлено {count:,} строк. В основном графике выбери «Добавленная станция» и даты из CSV.")
del batch

In [ ]:
# @title Скачать таблицу текущего выбора
export_view = "Исходные записи"  # @param ["Исходные записи", "Сутки", "Декады", "Месяцы"]
tables = {"Исходные записи": selected_rows, "Сутки": selected_daily.reset_index(),
          "Декады": selected_ten_day, "Месяцы": selected_monthly}
filename = f"eda_{station}_{metric}.csv"
tables[export_view].assign(station=archive.labels[station], metric=metric, unit=METRICS[metric][1]).to_csv(
    filename, index=False, encoding="utf-8-sig"
)
try:
    from google.colab import files
except ImportError:
    print(f"Таблица сохранена в {filename}")
else:
    files.download(filename)

## Свободный анализ в PyGWalker

Запусти этот раздел, если хочешь сама собирать графики. Для отдельных замеров выбери **Scatter**. Для суточных средних поставь `day` на X, а `mean` на Y. Вариант с исходными данными включает причины исключения. Для выборки свыше 25 000 строк используется равномерное прореживание, а кнопка скачивания отдаёт полную таблицу.

In [ ]:
%pip -q install "pygwalker==0.5.0.1"

In [ ]:
# @title Открыть выбранную таблицу в PyGWalker
import pygwalker as pyg

walker_view = "Пригодные замеры"  # @param ["Исходные записи", "Пригодные замеры", "Суточная сводка", "Месячное покрытие"]
walker_tables = {
    "Исходные записи": selected_rows,
    "Пригодные замеры": selected_rows.loc[selected_rows.valid],
    "Суточная сводка": selected_daily.reset_index(),
    "Месячное покрытие": selected_monthly,
}
data = walker_tables[walker_view]
if data.empty:
    print("В этой таблице нет записей. Выбери другую таблицу или период.")
else:
    step = max(1, (len(data) + 24999) // 25000)
    print(f"В PyGWalker показано {len(data.iloc[::step]):,} из {len(data):,} строк")
    pyg.walk(data.iloc[::step].reset_index(drop=True))

## Для защиты

1. Покажи исключённые замеры давления и полноту Кр1 по дням.
2. Открой 2 апреля и покажи исходный файл и флаг `pressure_hpa:range`.
3. Сравни среднюю разницу за июль и суточный ход поля и поляны.
4. Скачай таблицу выбранного сценария, чтобы сохранить воспроизводимый результат.

Расчёты основаны на описании кейса, `README.md` из архива и проверке по `coverage_by_month.csv`. [Plotly в Colab](https://plotly.com/python/renderers/) · [Документация PyGWalker](https://docs.kanaries.net/pygwalker)